In [1]:
from os import PathLike
from hdfs import InsecureClient
from pyspark.sql import SparkSession
from pyspark.sql import Row
from delta import *
from pyspark.sql.types import LongType, StringType, StructField, StructType, BooleanType, ArrayType, IntegerType

In [2]:
# warehouse_location points to the default location for managed databases and tables
warehouse_location = 'hdfs://hdfs-nn:9000/warehouse'

builder = SparkSession \
    .builder \
    .appName("Python Spark SQL Hive integration for the EDSTD project") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:1.0.0") \
    .enableHiveSupport() \

spark = spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [3]:
spark.sql(
    """
    SHOW TABLES FROM projeto
    """
).show()

+---------+---------------+-----------+
|namespace|      tableName|isTemporary|
+---------+---------------+-----------+
|  projeto|netflix_credits|      false|
+---------+---------------+-----------+



In [ ]:
# spark.sql(
#    """
#    DROP TABLE IF EXISTS projeto.amazon_credits
#    """
#)

In [4]:
spark.sql(
    """
    CREATE EXTERNAL TABLE projeto.amazon_credits (
        person_id INT,
        id VARCHAR(50),
        name VARCHAR(80),
        character VARCHAR(300)
    )
    
    USING DELTA
    PARTITIONED BY (
        role VARCHAR(50)
    )
    LOCATION 'hdfs://hdfs-nn:9000/warehouse/projeto.db/amazon_credits/'
    """
)

DataFrame[]

In [6]:
spark.sql(
    """
    DESCRIBE FORMATTED projeto.amazon_credits
    """
).toPandas()

,col_name,data_type,comment
0,person_id,int,None
1,id,string,None
2,name,string,None
3,character,string,None
4,role,string,None
5,# Partition Information,,
6,# col_name,data_type,comment
7,role,string,None
8,,,
9,# Detailed Table Information,,


In [10]:
spark.sql(
    """
    SELECT * FROM projeto.amazon_credits
    """
).show()

+---------+-------+-------------------+--------------------+-----+
|person_id|     id|               name|           character| role|
+---------+-------+-------------------+--------------------+-----+
|    59401|ts20945|         Joe Besser|                 Joe|ACTOR|
|    31460|ts20945|         Moe Howard|                 Moe|ACTOR|
|    31461|ts20945|         Larry Fine|               Larry|ACTOR|
|    21174|tm19248|      Buster Keaton|         Johnny Gray|ACTOR|
|    28713|tm19248|        Marion Mack|       Annabelle Lee|ACTOR|
|    28714|tm19248|      Glen Cavender|    Captain Anderson|ACTOR|
|    28715|tm19248|         Jim Farley|    General Thatcher|ACTOR|
|    27348|tm19248|    Frederick Vroom|  A Southern General|ACTOR|
|    28716|tm19248|Charles Henry Smith|  Annabelle's Father|ACTOR|
|    28718|tm19248|         Joe Keaton|       Union General|ACTOR|
|    28721|tm19248|        Al St. John|Officer on Horseb...|ACTOR|
|    28717|tm19248|       Frank Barnes| Annabelle's Brother|AC

In [11]:
spark.sql(
    """
    DESCRIBE HISTORY projeto.amazon_credits
    """
).show()

+-------+--------------------+------+--------+------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|   operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      1|2025-10-30 23:14:...|  null|    null|       WRITE|{mode -> Overwrit...|null|    null|     null|          0|  Serializable|        false|{numFiles -> 2, n...|        null|Apache-Spark/3.4....|
|      0|2025-10-30 23:14:...|  null|    null|CREATE TABLE|{isManaged -> fal...|null|    null|     null|       null|  Serializable|         true|                  {}|        null|Apache-Spark/3.4.